# 2026-09-21：新增 L2 数据接入检查

以下摘要对应 2026-09-21 隔离验证执行时；随后授权的发布见[发布记录](#data-publication-20260921)。

本轮只核对并隔离重建数据链路，不读取新日期的模型效果，不改变原实验失败结论或验收门槛。研究前提与结论唯一入口为 [README](README.md)。正式存储只读。

上次正常化因 4,187 条时段不匹配而失败，失败版本已归档。当前分支新增 SH 14:57 复牌集合竞价候选窗口；dev 尚未包含该修改。本次以候选代码重验同一输入，不声明正式采用。

**执行结果：5 个代码单元完成。SH 保留 63,405,970 条成交，4,187 条复牌成交均归为集合竞价；18 个日期的特征检查通过。**

另由归档 `prepare_new_panel.py` 沿用 H02 白名单装配了 17 个新目标日、88,506 行、39 个特征的数据面板，其中 88,395 行有可观察标签。未进行新选模或模型效果评价；17 日尚不足原 50 日最终验收门槛。

以上派生对象均保存在隔离研究目录，未发布到正式数据目录；候选时段规则尚未合入 dev。

## 证据说明索引

以下说明于 2026-09-24 从同主题 README 整理迁入，保存各节标注日期的方案、事实与当时解释；迁移不是新的运行或事前登记。当前假设、状态、结论和决定由 [README](README.md) 拥有。历史归档中的源码、Notebook、输入、验收快照和保留期继续有效，精确复现按对应归档说明执行。

- [2026-09-21 数据接入与随后授权的正式发布](#data-publication-20260921)


## 1. 归档源码与输入

已验证旧实验 1,094 份输入内容没有变化。当前原始成交最晚到 9 月 18 日，但 9 月 17 日下游存在缺口；先归档运行前方案和本轮实际输入。

In [1]:
import importlib.metadata
import json
import runpy
import shutil
import subprocess
import tarfile
import tempfile
import time
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

from src.access import Access, meta
from src.data_system.builders.level2_stock_trade_1m import (
    STOCK_TRADE_1M_KEY,
    build_level2_stock_trade_1m,
)
from src.data_system.builders.stock_1430 import (
    STOCK_1430_FEATURE_SCHEMA,
    STOCK_1430_KEY_SCHEMA,
    build_stock_1430_features,
    build_stock_1430_labels,
    require_stock_1430_feature_keys,
)
from src.data_system.normalize.level2 import normalize_level2
from src.utils.datetime_utils import DateTimeUtils
from src.utils.parquet_writer import write_parquet_atomic
from src.utils.path import PathManager

source = Path("/home/wsw/app/dev/trading")
origin_pm = PathManager(Path("/home/wsw/app/data"))
origin_access = Access(pm=origin_pm, processed_version="v1")
evidence = Path(
    tempfile.mkdtemp(
        prefix="subscription-data-update-2026-09-21-",
        dir="/home/wsw/app/research-evidence",
    )
)
print("evidence:", evidence, flush=True)
storage = evidence / "storage"
storage.mkdir()
pm = PathManager(storage)
access = Access(pm=pm, processed_version="v1")
shutil.copy2(
    source / "research/subscription-training/README.md",
    evidence / "preregistered-README.md",
)
shutil.copy2(
    source / "research/subscription-training/data_update_validation.ipynb",
    evidence / "unexecuted-notebook.ipynb",
)
shutil.copy2(
    source / "research/subscription-training/validate_training.py",
    evidence / "validate_training.py",
)
with tarfile.open(evidence / "source.tar", "w") as archive:
    for item in sorted((source / "src").rglob("*.py")):
        archive.add(item, arcname=item.relative_to(source))
    for name in ("pyproject.toml", "uv.lock"):
        archive.add(source / name, arcname=name)

helper = runpy.run_path(
    str(source / "research/subscription-training/validate_training.py")
)
sz_paths = origin_pm.processed_object(
    dataset_name="sz_trade", version="v1", trade_date="2026-09-17"
)
helper["_snapshot_object"](origin_pm, sz_paths, storage)
raw_dir = (
    origin_pm.storage_root / "raw/level2_ftp/sh_stock_ordertrade/trade_date=2026-09-17"
)
raw = meta.require(pm=origin_pm, meta_path=raw_dir / "meta.json")
for item in (raw_dir / "meta.json", raw.payload_path):
    destination = storage / item.relative_to(origin_pm.storage_root)
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(item, destination)
    assert helper["_sha256"](item) == helper["_sha256"](destination)
copied_inputs = [
    {
        "path": str(item.relative_to(storage)),
        "size_bytes": item.stat().st_size,
        "sha256": helper["_sha256"](item),
    }
    for item in sorted(storage.rglob("*"))
    if item.is_file()
]
(evidence / "copied-inputs.json").write_text(json.dumps(copied_inputs, indent=2))
run = {
    "base_commit": subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=source, text=True
    ).strip(),
    "worktree_status_before": subprocess.check_output(
        ["git", "status", "--short"], cwd=source, text=True
    ),
    "versions": {
        name: importlib.metadata.version(name)
        for name in ("numpy", "pandas", "pyarrow", "scikit-learn")
    },
    "source_storage_read_only": True,
    "alpha_evaluated": False,
}
(evidence / "run.json").write_text(json.dumps(run, ensure_ascii=False, indent=2))
print("copied input files:", len(copied_inputs), flush=True)

evidence: /home/wsw/app/research-evidence/subscription-data-update-2026-09-21-z06opybc


copied input files: 6


## 2. 重建上海 9 月 17 日逐笔

调用现有正常化规则，保留原始过滤、证券分类、市场时段和方向定义。异常必须原样停止，不能删除问题记录或放宽规则。

In [2]:
started = time.monotonic()
raw_copy = storage / raw.payload_path.relative_to(origin_pm.storage_root)
sh_paths = pm.processed_object(
    dataset_name="sh_trade", version="v1", trade_date="2026-09-17"
)
normalized = normalize_level2(
    input_file=raw_copy,
    output_name=sh_paths.payload_path,
    raw_object="SH_Stock_OrderTrade",
    trade_date="2026-09-17",
    target_name="sh_trade",
)
assert normalized.table.num_rows == 63_405_970
from datetime import time as wall_time

from src.data_system.market_phase import MarketPhase

resume_start = DateTimeUtils.local_time_to_utc_epoch_us(
    wall_time(14, 57), date(2026, 9, 17)
)
resume_rows = normalized.table.filter(
    pc.and_(
        pc.equal(normalized.table["symbol"], "601091"),
        pc.and_(
            pc.greater_equal(normalized.table["ts_utc"], resume_start),
            pc.less(normalized.table["ts_utc"], resume_start + 1_000_000),
        ),
    )
)
assert resume_rows.num_rows == 4187
assert pc.all(pc.equal(resume_rows["phase"], int(MarketPhase.AUCTION))).as_py()
write_parquet_atomic(
    output_file=evidence / "resume-auction-601091.parquet", table=resume_rows
)
write_parquet_atomic(output_file=sh_paths.payload_path, table=normalized.table)
meta.commit(
    pm=pm,
    payload_path=sh_paths.payload_path,
    upstream_meta_path=raw_copy.parent / "meta.json",
    symbol_slices=normalized.symbol_slices,
)
normalization_result = {
    "rows": normalized.table.num_rows,
    "symbols": len(normalized.symbol_slices),
    "elapsed_seconds": time.monotonic() - started,
}
(evidence / "normalization-result.json").write_text(
    json.dumps(normalization_result, indent=2)
)
del normalized
normalization_result

2026-09-21 11:55:18.018 | INFO     | src.data_system.normalize.level2:normalize_level2:165 - ▶️ Level-2 normalize; target=sh_trade trade_date=2026-09-17 input=SH_Stock_OrderTrade.csv.7z


2026-09-21 11:55:47.591 | INFO     | src.data_system.normalize.level2:normalize_level2:189 - ⏳ Level-2 normalize; target=sh_trade trade_date=2026-09-17 elapsed_seconds=30


2026-09-21 11:56:17.596 | INFO     | src.data_system.normalize.level2:normalize_level2:189 - ⏳ Level-2 normalize; target=sh_trade trade_date=2026-09-17 elapsed_seconds=60


2026-09-21 11:56:47.609 | INFO     | src.data_system.normalize.level2:normalize_level2:189 - ⏳ Level-2 normalize; target=sh_trade trade_date=2026-09-17 elapsed_seconds=90


2026-09-21 11:57:17.615 | INFO     | src.data_system.normalize.level2:normalize_level2:189 - ⏳ Level-2 normalize; target=sh_trade trade_date=2026-09-17 elapsed_seconds=120


...

2026-09-21 11:58:20.077 | INFO     | src.data_system.normalize.level2:_build_symbol_index:408 - ▶️ Level-2 sort; rows=63405970


2026-09-21 11:59:06.911 | INFO     | src.data_system.normalize.level2:_build_symbol_index:420 - ✅ Level-2 sort; rows=63405970


2026-09-21 11:59:07.195 | INFO     | src.data_system.normalize.level2:_build_symbol_index:432 - ✅ Level-2 symbol index; symbols=3459


2026-09-21 11:59:08.063 | INFO     | src.data_system.normalize.level2:normalize_level2:206 - ✅ Level-2 normalize; target=sh_trade trade_date=2026-09-17 rows=63405970 symbols=3459 elapsed_seconds=230 output=/home/wsw/app/research-evidence/subscription-data-update-2026-09-21-z06opybc/storage/processed/sh_trade/v1/trade_date=2026-09-17/data.parquet


{'rows': 63405970, 'symbols': 3459, 'elapsed_seconds': 246.73759856796823}

## 3. 重建两市分钟与 9 月 17 日 Feature

通过隔离存储中的 Access 分批读取，每个 symbol 完整进入既有分钟 builder。随后由 Access 检查两市分钟对象，再生成 Feature。

In [3]:
minute_results = []
for exchange in ("sh", "sz"):
    symbols = access.level2_symbols(trade_date="2026-09-17", exchange=exchange)
    parts = []
    for offset in range(0, len(symbols), 100):
        trades = access.trades(
            trade_date="2026-09-17",
            symbols=symbols[offset : offset + 100],
            exchange=exchange,
        )
        part = build_level2_stock_trade_1m(
            pa.concat_tables(list(trades.values())), trade_date="2026-09-17"
        )
        parts.append(part)
    minutes = pa.concat_tables(parts).sort_by(
        [(key, "ascending") for key in STOCK_TRADE_1M_KEY]
    )
    paths = pm.processed_object(
        dataset_name=f"{exchange}_stock_trade_1m", version="v1", trade_date="2026-09-17"
    )
    upstream = pm.processed_object(
        dataset_name=f"{exchange}_trade", version="v1", trade_date="2026-09-17"
    )
    write_parquet_atomic(output_file=paths.payload_path, table=minutes)
    meta.commit(
        pm=pm, payload_path=paths.payload_path, upstream_meta_path=upstream.meta_path
    )
    minute_results.append({"exchange": exchange, "minute_rows": minutes.num_rows})
    del trades, part, parts, minutes
minutes_17 = access.stock_trade_minutes(trade_date="2026-09-17")
feature_17 = build_stock_1430_features(minutes_17, trade_date=date(2026, 9, 17))
feature_path = pm.feature_object(
    feature_set="l2_stock_1430", version="v1", trade_date="2026-09-17"
)
write_parquet_atomic(output_file=feature_path.payload_path, table=feature_17)
meta.commit(pm=pm, payload_path=feature_path.payload_path)
pd.DataFrame(minute_results)

,exchange,minute_rows
0,sh,535702
1,sz,678806


## 4. 恢复 9 月 16、17 日监督目标

使用已有 T+1 窗口规则及真实复权因子。只保存标签和覆盖，不读取策略排名表现；9 月 18 日的标签仍需下一个交易日的数据。

In [4]:
label_results = []
for target in ("2026-09-16", "2026-09-17"):
    following = origin_access.next_trade_date(trade_date=target)
    assert following == ("2026-09-17" if target == "2026-09-16" else "2026-09-18")
    if target == "2026-09-17":
        feature = feature_17
        entry = minutes_17
        exit_minutes = origin_access.stock_trade_minutes(trade_date=following)
        write_parquet_atomic(
            output_file=evidence / f"input-minutes-{following}.parquet",
            table=exit_minutes,
        )
    else:
        paths = origin_pm.feature_object(
            feature_set="l2_stock_1430", version="v1", trade_date=target
        )
        helper["_snapshot_object"](origin_pm, paths, storage)
        feature = helper["_read_object"](
            pm,
            pm.feature_object(
                feature_set="l2_stock_1430", version="v1", trade_date=target
            ),
        )
        feature = pa.Table.from_pandas(
            feature, schema=STOCK_1430_FEATURE_SCHEMA, preserve_index=False
        )
        entry = origin_access.stock_trade_minutes(trade_date=target)
        write_parquet_atomic(
            output_file=evidence / f"input-minutes-{target}.parquet", table=entry
        )
        exit_minutes = minutes_17
    factors_t = origin_access.adjustment_factors(trade_date=target)
    factors_t1 = origin_access.adjustment_factors(trade_date=following)
    for day, factors in ((target, factors_t), (following, factors_t1)):
        destination = evidence / f"input-factors-{day}.parquet"
        if not destination.exists():
            write_parquet_atomic(
                output_file=destination,
                table=pa.Table.from_pandas(factors, preserve_index=False),
            )
    label = build_stock_1430_labels(
        feature_keys=feature.select(STOCK_1430_KEY_SCHEMA.names),
        entry_minutes=entry,
        exit_minutes=exit_minutes,
        entry_factors=factors_t,
        exit_factors=factors_t1,
        trade_date=date.fromisoformat(target),
        next_trade_date=date.fromisoformat(following),
    )
    assert label.select(STOCK_1430_KEY_SCHEMA.names).equals(
        feature.select(STOCK_1430_KEY_SCHEMA.names)
    )
    paths = pm.label_object(
        label_set="l2_stock_1430_t1_vwap_rank", version="v1", trade_date=target
    )
    write_parquet_atomic(output_file=paths.payload_path, table=label)
    meta.commit(pm=pm, payload_path=paths.payload_path)
    label_results.append(
        {
            "trade_date": target,
            "rows": label.num_rows,
            "missing_labels": label["y_rank_return"].null_count,
        }
    )
pd.DataFrame(label_results)

,trade_date,rows,missing_labels
0,2026-09-16,5205,8
1,2026-09-17,5208,5


## 5. 新日期特征的接入校验

检查日期、完整 key、14:30 决策时点、schema 和特征数值域。独立验证期仍需事前固定候选与通过标准；本轮不读取这些日期的模型效果。

In [5]:
from datetime import time as wall_time

sessions = origin_access.trade_dates(start_date="2026-08-26", end_date="2026-09-18")
feature_checks = []
for day in sessions:
    if day == "2026-09-17":
        table = feature_17
    else:
        paths = origin_pm.feature_object(
            feature_set="l2_stock_1430", version="v1", trade_date=day
        )
        helper["_snapshot_object"](origin_pm, paths, storage)
        record = meta.require(
            pm=origin_pm,
            meta_path=paths.meta_path,
            expected_payload_path=paths.payload_path,
        )
        with pq.ParquetFile(record.payload_path) as parquet:
            table = parquet.read()
    assert table.schema.equals(STOCK_1430_FEATURE_SCHEMA)
    require_stock_1430_feature_keys(
        table.select(STOCK_1430_KEY_SCHEMA.names),
        trade_date=day,
        decision_ts_utc=DateTimeUtils.local_time_to_utc_epoch_us(
            wall_time(14, 30), date.fromisoformat(day)
        ),
    )
    numbers = (
        table.select(STOCK_1430_FEATURE_SCHEMA.names[3:])
        .to_pandas()
        .to_numpy(dtype=np.float64, na_value=np.nan)
    )
    present = numbers[~np.isnan(numbers)]
    assert np.isfinite(present).all() and ((present >= 0) & (present <= 1)).all()
    feature_checks.append(
        {
            "trade_date": day,
            "rows": table.num_rows,
            "schema_key_values_valid": True,
            "origin": "isolated reconstruction"
            if day == "2026-09-17"
            else "committed source",
        }
    )
summary = {
    "new_feature_sessions_checked": len(feature_checks),
    "reconstructed_date": "2026-09-17",
    "available_new_target_dates_after_reconstruction": 17,
    "minimum_final_dates_in_H02": 50,
    "alpha_evaluated": False,
    "real_reception_readiness_verified": False,
    "source_storage_written": False,
    "normalization": normalization_result,
    "minute_outputs": minute_results,
    "label_outputs": label_results,
}
(evidence / "feature-checks.json").write_text(json.dumps(feature_checks, indent=2))
(evidence / "summary.json").write_text(json.dumps(summary, indent=2))
for item in copied_inputs:
    original = origin_pm.storage_root / item["path"]
    assert helper["_sha256"](original) == item["sha256"]
print("summary:", json.dumps(summary, ensure_ascii=False, indent=2))
pd.DataFrame(feature_checks)

summary: {
  "new_feature_sessions_checked": 18,
  "reconstructed_date": "2026-09-17",
  "available_new_target_dates_after_reconstruction": 17,
  "minimum_final_dates_in_H02": 50,
  "alpha_evaluated": false,
  "real_reception_readiness_verified": false,
  "source_storage_written": false,
  "normalization": {
    "rows": 63405970,
    "symbols": 3459,
    "elapsed_seconds": 246.73759856796823
  },
  "minute_outputs": [
    {
      "exchange": "sh",
      "minute_rows": 535702
    },
    {
      "exchange": "sz",
      "minute_rows": 678806
    }
  ],
  "label_outputs": [
    {
      "trade_date": "2026-09-16",
      "rows": 5205,
      "missing_labels": 8
    },
    {
      "trade_date": "2026-09-17",
      "rows": 5208,
      "missing_labels": 5
    }
  ]
}


,trade_date,rows,schema_key_values_valid,origin
0,2026-08-26,5208,True,committed source
1,2026-08-27,5208,True,committed source
2,2026-08-28,5207,True,committed source
3,2026-08-31,5205,True,committed source
4,2026-09-01,5206,True,committed source
5,2026-09-02,5207,True,committed source
6,2026-09-03,5208,True,committed source
7,2026-09-04,5206,True,committed source
8,2026-09-07,5206,True,committed source
9,2026-09-08,5206,True,committed source


<a id="data-publication-20260921"></a>

## 2026-09-21 数据接入与随后授权的正式发布

本 Notebook 的代码及输出对应隔离验证；随后发布由归档中的独立脚本完成。下列记录区分两次操作，不以本 Notebook 的执行证明正式发布。

用户说明数据已更新后核对发现：raw/部分下游最晚至9月18日，但9月17日缺 SH 逐笔、
两市分钟及 H03 Feature，Label 仅至9月15日。原 H02 的1,094份输入摘要未变。
预先固定在隔离副本重建9月17日及9月16/17日 Label，验证全链、schema/key/时间/数值域
和连续覆盖；失败保留，不在这一步训练或查看新增日期效果。

基线 `82999e8bd4a6be15ac5070db6247cf976795bb04` 的 SH 正常化因4,187条成交未匹配时段
失败，证据在 `/home/wsw/app/research-evidence/subscription-data-update-2026-09-21-lgrtwnjn/`。
随后 raw/Meta 未变，但 feature 分支 `156e904a1e1b6b9aff1ed8d5fffb3271b19819ee`
已加入 SH stock/cdr 的 `[14:57:00,14:57:01)` 复牌集合竞价候选窗口；当时目标 dev
`a5021df6a8367aaf43e6e149847b18a5318bd25b` 尚无该窗口，故这里只证明候选实现通过。

[`data_update_validation.ipynb`](data_update_validation.ipynb) 最终全链完成，601091的4,187条
均为 AUCTION；18日 Feature 完整。新增面板 `08-26..09-17` 为17日、88,506个股票日、
39列特征、88,395个已知标签及111个缺失，保留原 P 22:00假设且未删未来标签缺失行。
中间研究校验曾因 Pandas 可空浮点 `to_numpy()` 得到 object 数组而使 `np.isnan` 失败；
明确转为 float64/NaN 后同验收重跑，13份关键派生文件逐字节相同，回归127 passed。

- 最终归档：`/home/wsw/app/research-evidence/subscription-data-update-2026-09-21-z06opybc/`，
  含源码、运行前方案、输入/派生对象、Notebook、`new-panel.parquet`、面板摘要及
  `prepare_new_panel.py`；保留至用户决定及审查结束。
- 转换错误归档：`/home/wsw/app/research-evidence/subscription-data-update-2026-09-21-675mrub9/`。

用户随后明确要求“将数据放置正式分区”。确认目标不存在、输入与验证版本一致后，
按上述 `156e904...` 代码发布以下六对象至 `/home/wsw/app/data`：

| 正式对象 | 日期 | 行数 |
| --- | --- | ---: |
| `processed/sh_trade/v1` | 2026-09-17 | 63,405,970 |
| `processed/sh_stock_trade_1m/v1` | 2026-09-17 | 535,702 |
| `processed/sz_stock_trade_1m/v1` | 2026-09-17 | 678,806 |
| `features/l2_stock_1430/v1` | 2026-09-17 | 5,208 |
| `labels/l2_stock_1430_t1_vwap_rank/v1` | 2026-09-16 | 5,205 |
| `labels/l2_stock_1430_t1_vwap_rank/v1` | 2026-09-17 | 5,208 |

payload 原子复制后 `meta.commit`，direct upstream/slices 保持原义；六份 payload 摘要和
六份 Meta 与验证产物相同。正式 Access 验证3,459个 SH slice、两市1,214,508行分钟，
重算 Feature/Label 完全一致；18日逐笔/分钟/Feature及17日 Label 正式读取检查通过。
发布归档 `/home/wsw/app/research-evidence/subscription-data-publication-2026-09-21-MBpp5QUo/`
保存 `publish.py`、`preflight.json`、`published.json` 和 `result.json`。授权仅针对这六个
数据对象，未采用模型/策略，也未合入时段候选或执行代码 commit、merge、release、deploy。
